# Русскоязычный корректор опечаток на основе шумового канала

**Задача:** исправлять отдельные опечатки, объединяя расстояние редактирования и вероятности слов в контексте.

**Стек:** Hugging Face Datasets, pandas, NumPy, `python-Levenshtein`.

Словарь и частоты строятся только по обучающей части корпуса; качество измеряется на отложенных парах `original → corrected`.


In [ ]:
import math
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from datasets import load_dataset
from Levenshtein import distance as levenshtein_distance
from sklearn.model_selection import train_test_split

SEED = 42
MAX_ROWS = 2_000
TOKEN_RE = re.compile(r"[а-яё]+|[^\w\s]", re.IGNORECASE)


In [ ]:
raw = load_dataset("keramjan/spellcheck", split=f"train[:{MAX_ROWS}]")
data = pd.DataFrame(raw)[["original", "corrected"]].dropna().drop_duplicates()
train_data, test_data = train_test_split(data, test_size=0.20, random_state=SEED)


def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(str(text).lower())


unigram_counts = Counter()
bigram_counts = Counter()
vocabulary = set()

for text in train_data["corrected"]:
    tokens = tokenize(text)
    words = [token for token in tokens if token.isalpha()]
    unigram_counts.update(words)
    bigram_counts.update(zip(["<s>"] + words[:-1], words))
    vocabulary.update(words)

prefix_index = defaultdict(set)
for word in vocabulary:
    prefix_index[word[:1]].add(word)
    prefix_index[word[:2]].add(word)

print(f"Обучающих пар: {len(train_data):,}")
print(f"Тестовых пар: {len(test_data):,}")
print(f"Размер словаря: {len(vocabulary):,}")


In [ ]:
def candidates(word: str, max_distance: int = 2, limit: int = 30) -> list[tuple[str, int]]:
    if word in vocabulary:
        return [(word, 0)]

    pool = prefix_index.get(word[:2], set()) | prefix_index.get(word[:1], set())
    ranked = []
    for candidate in pool:
        if abs(len(candidate) - len(word)) > max_distance:
            continue
        distance = levenshtein_distance(word, candidate)
        if distance <= max_distance:
            ranked.append((candidate, distance))

    return sorted(
        ranked,
        key=lambda item: (item[1], -unigram_counts[item[0]], item[0]),
    )[:limit]


def contextual_cost(previous: str, word: str) -> float:
    vocabulary_size = max(len(vocabulary), 1)
    numerator = bigram_counts[(previous, word)] + 1
    denominator = unigram_counts[previous] + vocabulary_size
    return -math.log(numerator / denominator)


def choose_candidate(word: str, previous: str) -> str:
    variants = candidates(word)
    if not variants:
        return word

    def score(item: tuple[str, int]) -> float:
        candidate, distance = item
        frequency_bonus = math.log1p(unigram_counts[candidate])
        return 1.2 * distance + contextual_cost(previous, candidate) - 0.15 * frequency_bonus

    best_word, best_distance = min(variants, key=score)
    return best_word if best_distance <= 2 else word


In [ ]:
def restore_spacing(tokens: list[str]) -> str:
    text = " ".join(tokens)
    return re.sub(r"\s+([,.;:!?])", r"\1", text)


def correct_text(text: str) -> str:
    tokens = tokenize(text)
    result = []
    previous_word = "<s>"

    for token in tokens:
        if token.isalpha():
            corrected = choose_candidate(token, previous_word)
            result.append(corrected)
            previous_word = corrected
        else:
            result.append(token)
    return restore_spacing(result)


def sequence_distance(left: list[str], right: list[str]) -> int:
    previous = list(range(len(right) + 1))
    for i, left_item in enumerate(left, start=1):
        current = [i]
        for j, right_item in enumerate(right, start=1):
            current.append(
                min(
                    previous[j] + 1,
                    current[j - 1] + 1,
                    previous[j - 1] + (left_item != right_item),
                )
            )
        previous = current
    return previous[-1]


In [ ]:
evaluation = test_data.head(100).copy()
evaluation["prediction"] = evaluation["original"].map(correct_text)

total_word_errors = 0
total_reference_words = 0
total_char_errors = 0
total_reference_chars = 0

for row in evaluation.itertuples(index=False):
    reference_words = tokenize(row.corrected)
    prediction_words = tokenize(row.prediction)
    total_word_errors += sequence_distance(reference_words, prediction_words)
    total_reference_words += len(reference_words)
    total_char_errors += levenshtein_distance(row.corrected.lower(), row.prediction.lower())
    total_reference_chars += len(row.corrected)

metrics = pd.Series(
    {
        "WER ↓": total_word_errors / max(total_reference_words, 1),
        "CER ↓": total_char_errors / max(total_reference_chars, 1),
        "Exact match ↑": np.mean(
            evaluation["corrected"].str.lower() == evaluation["prediction"].str.lower()
        ),
    }
)
metrics.round(4).to_frame("value")


In [ ]:
pd.set_option("display.max_colwidth", 120)
evaluation[["original", "corrected", "prediction"]].head(10)


## Ограничения

Модель компактна и интерпретируема, но не обрабатывает сложные грамматические ошибки и слова вне словаря. Следующий шаг — заменить биграмную модель на контекстный masked language model и сравнить качество и задержку.
